In [0]:
import time
from datetime import datetime

dbutils.widgets.text("github_token", "")
dbutils.widgets.text("repos", "langchain-ai/langchain,run-llama/llama_index,chroma-core/chroma,qdrant/qdrant,deepset-ai/haystack")
dbutils.widgets.text("start_date", "2024-10-10")
dbutils.widgets.text("end_date", "2026-10-10")
dbutils.widgets.text("load_type", "full")
dbutils.widgets.text("endpoints", "commits,issues")
dbutils.widgets.text("max_pages", "0")          # 0 = pull everything

g = lambda k: dbutils.widgets.get(k)
common = {"load_type": g("load_type"), "start_date": g("start_date"), "end_date": g("end_date")}
failed = []

# 1) Source -> Raw, one repo at a time (so a failure only costs one repo)
for repo in [r.strip() for r in g("repos").split(",")]:
    t = time.time()
    try:
        dbutils.notebook.run("./01_ingest_raw", 0, {**common,
            "repos": repo, "github_token": g("github_token"),
            "endpoints": g("endpoints"), "max_pages": g("max_pages")})
        print(f"OK   {repo} ({int(time.time()-t)}s)")
    except Exception as e:
        failed.append(repo)
        print(f"FAIL {repo}: {str(e)[:200]}")

# 2) Raw -> Bronze (skips files already loaded)
dbutils.notebook.run("./02_raw_to_bronze", 0, {"load_type": g("load_type"), "ingest_date": "*"})

# 3) Bronze -> Silver (MERGE, safe to rerun)
dbutils.notebook.run("./03_bronze_to_silver", 0, {"load_type": g("load_type")})

print("Done. Failed repos:", failed or "none")
display(spark.sql("""SELECT layer, status, SUM(rows_inserted) AS inserted, SUM(rows_updated) AS updated
                     FROM workspace.rag.pipeline_execution_logs GROUP BY layer, status ORDER BY layer"""))